# Helix — End-to-end smoke tests

Manual smoke tests against a running Helix server (`make dev`, default port 8003).

Each cell hits the live `/check` endpoint, which runs the full agentic pipeline:
the FastAPI layer ingests the URL (deterministic Python), the Orchestrator LLM
extracts the main claim, then transfers to the Investigator LLM which fact-checks
via PubMed + WHO + Google Fact Check Tools.

Tests are independent — run any subset.

1. **Plain text claim** — direct fact-check (no ingestion, no claim extraction)
2. **Question reformulation** — `"Does X cure Y?"` → claim `"X cures Y"`
3. **Article URL** — Doctissimo wellness blog (French, multilingual handling)
4. **YouTube URL** — TED-Ed video with captions
5. **TikTok URL** — audio-only path via on-device Gemma 4 (requires ffmpeg + HF login)

Bonus cell at the end smokes `/ingest` alone (the fast deterministic step the UI
calls before `/check` to render the source in the loading panel).

In [ ]:
import json, time, textwrap
import requests

BASE = "http://localhost:8003"
TIMEOUT = 600  # the agentic pipeline can take 1-2 min for URLs

BAND_EMOJI = {
    "Supported": "✅", "Partially supported": "🟡",
    "Insufficient evidence": "⚪", "Contradicted": "❌",
    "Known misinformation": "🚫",
}

def post_check(payload, label):
    """Hit POST /check, print a tidy summary, return the raw JSON."""
    print(f"▶  {label}")
    print(f"   payload: {json.dumps(payload)[:140]}")
    started = time.perf_counter()
    r = requests.post(f"{BASE}/check", json=payload, timeout=TIMEOUT)
    elapsed = time.perf_counter() - started
    print(f"   HTTP {r.status_code} · server elapsed: {r.json().get('elapsed_seconds') if r.ok else '—'}s · client: {elapsed:.1f}s")
    if not r.ok:
        print("  ", r.text[:1500])
        return None
    data = r.json()

    results = data.get("results", [])
    if not results:
        print("   ⚠️  no claim identified")
        return data

    for i, res in enumerate(results, 1):
        claim = res["claim"]
        v = res["verdict"]
        emoji = BAND_EMOJI.get(v["band"], "❓")
        print(f"\n   --- Claim {i} ---")
        print(f"   {emoji} {v['band']} (score {v['score']})")
        print(f"   claim ({claim.get('language', '?')}, {claim.get('domain', '?')}): {claim['text']}")
        for f in v.get("findings", []):
            src_count = len(f.get("sources") or [])
            print(f"     • {f['agent']:12s} stance={f['stance']:12s} confidence={f['confidence']:6s} sources={src_count}")
        if v.get("narrative"):
            print("   narrative: " + textwrap.shorten(v["narrative"], 320))
        if v.get("overall_assessment"):
            print("   take:      " + textwrap.shorten(v["overall_assessment"], 320))
    return data

# Sanity check the server is up.
print(requests.get(f"{BASE}/health", timeout=5).json())

## Test 1 — Plain text claim

User types a finished claim. No ingestion, no claim extraction — the orchestrator
passes the claim straight through to the investigator. Fastest path (~60-80s).

Expected: `Contradicted` or `Known misinformation`, citing PubMed + WHO + a fact-check.

In [ ]:
post_check(
    {"text": "Drinking lemon water on an empty stomach cures stage IV cancer."},
    label="Test 1 — plain text claim",
)
print("\n⏸  cooldown 30s before next test"); time.sleep(30)

## Test 2 — Question reformulation

User asks a question. The Orchestrator must rewrite it as a positive testable
claim (e.g. *"Does vitamin C cure COVID-19?"* → *"Vitamin C cures COVID-19."*)
before transferring to the investigator.

In [ ]:
post_check(
    {"text": "Is it true that vitamin C cures COVID-19?"},
    label="Test 2 — question reformulation",
)
print("\n⏸  cooldown 30s"); time.sleep(30)

## Test 3 — Article URL (French)

Tests `trafilatura` article extraction + multilingual handling. The orchestrator
should pick the main claim from a long French health article, translate it to
English (so PubMed can be queried), and keep the original language code so the
narrative comes back in French.

In [ ]:
post_check(
    {"url": "https://www.doctissimo.fr/sante/cholesterol"},
    label="Test 3 — Doctissimo article (FR)",
)
print("\n⏸  cooldown 30s"); time.sleep(30)

## Test 4 — YouTube video with captions

Tests `youtube-transcript-api` ingestion (no LLM transcription — captions only).
Picks a TED-Ed video on sugar and the body. If captions are disabled on the
video, `/check` returns HTTP 422 with an explicit error — swap the URL for
another video that has captions.

In [ ]:
post_check(
    {"url": "https://www.youtube.com/watch?v=78CJOrZ1Z2c"},
    label="Test 4 — YouTube video (captions)",
)
print("\n⏸  cooldown 30s"); time.sleep(30)

## Test 5 — TikTok video

Tests the on-device Gemma 4 audio path. Requirements:
- `ffmpeg` on PATH (`brew install ffmpeg` if missing)
- HuggingFace login with Gemma 4 license accepted
- Model weights downloaded on first run (~16 GB, lazy)

Skip this cell if any of those is not set up locally. TikTok URLs rot fast —
replace the URL below with any current TikTok health short.

In [ ]:
post_check(
    {"url": "https://www.tiktok.com/@drsoodmd/video/7407094892503944491"},
    label="Test 5 — TikTok video (audio path)",
)

## Bonus — `/ingest` smoke test

Mirrors what the frontend does *before* `/check`: a fast, deterministic call that
returns the ingested text so the loading panel has something to render while the
agent crunches. No LLM involved — should respond in 1-5s for an article.

In [ ]:
r = requests.post(
    f"{BASE}/ingest",
    json={"url": "https://www.youtube.com/watch?v=78CJOrZ1Z2c"},
    timeout=60,
)
r.raise_for_status()
ing = r.json()
print("kind:        ", ing["kind"])
print("source_url:  ", ing["source_url"])
print("title:       ", ing["title"])
print("text length: ", len(ing["text"]))
print("first 400:   ", ing["text"][:400])